In [12]:
"""
Reconstruction from embeddings experiment
With codecs destroyed
"""


import torch
from dataclasses import dataclass
from pathlib import Path
from voice_to_audio import load_decoder_and_embtable, voice_embedding_to_audio, save_wav, get_codes

@dataclass
class Args:
    dtype: str = 'float16'
    device: str = 'cpu'
    model: str = "../voxtral-tts-weights"
    voice: str = "../voice_embeddings/casual_female.pt"
    max_iters: int = 10
    output: str = "casual_female_corrupted.wav"

def get_voice_embedding(voice_path: Path):
    voice_emb: torch.Tensor = torch.load(
            str(voice_path), map_location="cpu", weights_only=True
        ).float()
    return voice_emb

args = Args()
dtype_map = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}
dtype  = dtype_map[args.dtype]
device = args.device
model_path = Path(args.model)



# Load decoder weights + embedding table (once, shared across all files)
tokenizer, emb_table = load_decoder_and_embtable(model_path, dtype=dtype, device=device)

voice_emb = get_voice_embedding(Path(args.voice))
codes = get_codes(voice_emb, emb_table, args.max_iters, args.device)

def corrupt_codes(codes):
    t, f = codes.shape
    codes = torch.cat((codes, torch.tensor([1, *[0]*(f-1)]).unsqueeze(0)), dim=0)
    codes = codes.transpose(0,1)
    codes[0] = torch.cat((torch.randint(2,8192,(codes.shape[1]-1,)), torch.tensor([1])), dim=0) #torch.tensor([3]*codes.shape[1])
    codes[1] = torch.cat((torch.randint(2,21,(codes.shape[1]-1,)), torch.tensor([0])), dim=0) #torch.tensor([3]*codes.shape[1])
    codes[2] = torch.cat((torch.randint(2,21,(codes.shape[1]-1,)), torch.tensor([0])), dim=0)
    codes[3] = torch.cat((torch.randint(2,21,(codes.shape[1]-1,)), torch.tensor([0])), dim=0)
    codes = codes.transpose(0,1)
    return codes
codes = corrupt_codes(codes)

# ---- Single file ----
if args.voice:
    voice_path = Path(args.voice)
    waveform, dur = voice_embedding_to_audio(
        tokenizer, emb_table, args.max_iters, device = device, voice_emb = voice_emb, codes = codes
    )
    out = Path(args.output) if args.output else voice_path.with_suffix(".wav")
    save_wav(out, waveform)
    print(f"Duration: {dur:.2f}s  |  Samples: {len(waveform)}  |  Saved: {out}")

Skipping reconstruction check



────────────────────────────────────────────────────────────
Voice:   |  215 frames  (17.20s @ 12.5Hz)
────────────────────────────────────────────────────────────
  Codes shape:    [215, 37]  (T frames × 37 codebooks)
  Semantic  (cb0): [1021, 3985, 1979, 3698, 5441, 5261, 2143, 6694, 4837, 1786] ...
  Acoustic0 (cb1): [16, 12, 14, 5, 17, 18, 2, 9, 15, 14] ...
  Semantic range: [1, 8182]  (0=EMPTY, 1=END, 2..=8193)
  Acoustic range: [0, 22]  (0=EMPTY, 1=END, 2..=22)
  END_AUDIO frames: 1
────────────────────────────────────────────────────────────

Duration: 17.12s  |  Samples: 410880  |  Saved: casual_female_corrupted.wav


In [11]:
"""
Reconstruction from embeddings experiment
With codecs destroyed
"""


import torch
from dataclasses import dataclass
from pathlib import Path
from voice_to_audio import load_decoder_and_embtable, voice_embedding_to_audio, save_wav, get_codes

@dataclass
class Args:
    dtype: str = 'float16'
    device: str = 'cpu'
    model: str = "../voxtral-tts-weights"
    voice: str = "../voice_embeddings/casual_female.pt"
    max_iters: int = 10
    output: str = "casual_female_clean.wav"

def get_voice_embedding(voice_path: Path):
    voice_emb: torch.Tensor = torch.load(
            str(voice_path), map_location="cpu", weights_only=True
        ).float()
    return voice_emb

args = Args()
dtype_map = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}
dtype  = dtype_map[args.dtype]
device = args.device
model_path = Path(args.model)



# Load decoder weights + embedding table (once, shared across all files)
tokenizer, emb_table = load_decoder_and_embtable(model_path, dtype=dtype, device=device)

voice_emb = get_voice_embedding(Path(args.voice))
codes = get_codes(voice_emb, emb_table, args.max_iters, args.device)


# ---- Single file ----
if args.voice:
    voice_path = Path(args.voice)
    waveform, dur = voice_embedding_to_audio(
        tokenizer, emb_table, args.max_iters, device = device, voice_emb = voice_emb, codes = codes
    )
    out = Path(args.output) if args.output else voice_path.with_suffix(".wav")
    save_wav(out, waveform)
    print(f"Duration: {dur:.2f}s  |  Samples: {len(waveform)}  |  Saved: {out}")

[]  RelErr 0.0136 is high — audio quality may be degraded.



────────────────────────────────────────────────────────────
Voice:   |  214 frames  (17.12s @ 12.5Hz)
────────────────────────────────────────────────────────────
  Codes shape:    [214, 37]  (T frames × 37 codebooks)
  Semantic  (cb0): [1862, 7376, 7946, 799, 349, 326, 4708, 3603, 7575, 3153] ...
  Acoustic0 (cb1): [2, 5, 3, 11, 10, 3, 11, 9, 10, 12] ...
  Semantic range: [1, 8099]  (0=EMPTY, 1=END, 2..=8193)
  Acoustic range: [0, 22]  (0=EMPTY, 1=END, 2..=22)
  END_AUDIO frames: 1
────────────────────────────────────────────────────────────

Duration: 17.04s  |  Samples: 408960  |  Saved: casual_female_clean.wav


In [20]:
codes

tensor([[2438,    4,   18,   10,    7,   17,   10,    5,    6,   16,    8,   15,
            2,    5,   16,    9,   14,   13,    8,   15,    6,    9,    6,    3,
           14,    6,    8,   12,   22,   13,   10,    5,   13,    4,   14,    6,
            5],
        [6889,    8,    8,    5,    7,   13,   15,    4,    9,   19,    6,   11,
            8,    8,    8,    8,    7,    9,    4,   12,   12,   17,    7,    5,
            9,   16,   20,   17,   22,   14,   10,    3,   14,    4,   15,    5,
            5],
        [2481,    5,   12,    6,    3,   10,   11,    6,   12,   10,    8,   16,
            4,   10,   15,   17,   12,   15,   11,   16,    7,   18,    9,    4,
           14,   16,   15,    9,   22,   13,   15,    3,    9,   11,   20,    5,
            3],
        [2204,   15,   15,   10,    6,   10,   11,    3,    9,    6,    6,   13,
            8,   10,   16,   16,    9,   12,    3,   18,    8,    7,   11,    3,
            7,   11,   12,    7,   21,   16,   15,    5,   16

In [6]:
codes.shape

torch.Size([147, 37])

In [1]:
# Load final codes after voice was reconstructed
import torch

torch.load("checkpoints/final_codes.pt")

tensor([[5627,    0,   18,  ...,   18,    1,    0],
        [3347,    5,    0,  ...,    7,    4,    3],
        [3169,    0,    3,  ...,    8,    2,    0],
        ...,
        [3950,    2,   20,  ...,   11,    5,    3],
        [ 563,    1,   20,  ...,   19,    2,    0],
        [1192,   20,   20,  ...,   20,    5,    0]])